# Day 32: Human-in-the-Loop

Day 31's agent called tools on its own. No permission. No oversight.

That's fine for weather lookups. But what about deleting data or sending emails? Today we add a human checkpoint.

## Install

In [ ]:
%pip install langgraph langchain-google-genai langchain --quiet

## Setup

In [20]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
import requests

load_dotenv(dotenv_path='../.env')
os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY2"]

model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

HEADERS = {"User-Agent": "genai-50-days-demo/1.0"}

@tool
def get_weather(city: str) -> str:
    """Get current weather for a city."""
    resp = requests.get(f"https://wttr.in/{city}?format=j1", headers=HEADERS)
    data = resp.json()["current_condition"][0]
    return f"{city}: {data['temp_C']}°C, {data['weatherDesc'][0]['value']}"

tools = [get_weather]
print("✅ Model and tool ready")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Model and tool ready


## Build Graph with Approval

Same graph as Day 31. One change: `tool_node` now calls `interrupt()` before executing any tool.

In [21]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langchain_core.messages import ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver

class State(TypedDict):
    messages: Annotated[list, add_messages]

model_with_tools = model.bind_tools(tools)

def llm_node(state: State):
    return {"messages": [model_with_tools.invoke(state["messages"])]}

In [22]:
def tool_node(state: State):
    """Ask human for approval before executing each tool call."""
    results = []
    for call in state["messages"][-1].tool_calls:
        decision = interrupt({
            "tool": call["name"],
            "args": call["args"],
            "message": "Approve this tool call? (yes/no)"
        })
        if decision == "yes":
            fn = {t.name: t for t in tools}[call["name"]]
            output = fn.invoke(call["args"])
            results.append(ToolMessage(content=str(output), tool_call_id=call["id"]))
        else:
            results.append(ToolMessage(content="Tool call rejected by human.", tool_call_id=call["id"]))
    return {"messages": results}

print("✅ tool_node now requires human approval")

✅ tool_node now requires human approval


In [23]:
def should_continue(state: State) -> Literal["tool_node", "__end__"]:
    return "tool_node" if state["messages"][-1].tool_calls else END

builder = StateGraph(State)
builder.add_node("llm_node", llm_node)
builder.add_node("tool_node", tool_node)
builder.add_edge(START, "llm_node")
builder.add_conditional_edges("llm_node", should_continue, ["tool_node", END])
builder.add_edge("tool_node", "llm_node")

# Checkpointer is REQUIRED for interrupt — graph must save state to resume
agent = builder.compile(checkpointer=InMemorySaver())
print("✅ Agent with human approval compiled")

✅ Agent with human approval compiled


## Test: Approve

In [24]:
config = {"configurable": {"thread_id": "approve-test"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Berlin?"}]},
    config
)
print("⏸️ Graph paused! Agent wants to call:")
print(result["messages"][-1].tool_calls[0])

⏸️ Graph paused! Agent wants to call:
{'name': 'get_weather', 'args': {'city': 'Berlin'}, 'id': '6c873e09-84ce-4fe0-9d98-fb37e86fca75', 'type': 'tool_call'}


In [25]:
# Human approves — resume execution
result = agent.invoke(Command(resume="yes"), config)
print("✅ Approved! Agent says:")
print(result["messages"][-1].content)

✅ Approved! Agent says:
The weather in Berlin is 8°C with a light rain shower.


## Test: Reject

In [26]:
config2 = {"configurable": {"thread_id": "reject-test"}}

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in Tokyo?"}]},
    config2
)
print("⏸️ Graph paused! Agent wants to call:")
print(result["messages"][-1].tool_calls[0])

⏸️ Graph paused! Agent wants to call:
{'name': 'get_weather', 'args': {'city': 'Tokyo'}, 'id': 'bc07283a-6ac2-4b7d-aea3-b978e08eb0ec', 'type': 'tool_call'}


In [27]:
# Human rejects — tool never executes
result = agent.invoke(Command(resume="no"), config2)
print("❌ Rejected! Agent says:")
print(result["messages"][-1].content)

❌ Rejected! Agent says:
I'm sorry, I cannot fulfill this request. My tool access was rejected.


## Key Takeaways

1. **`interrupt()`** pauses the graph and surfaces info to the human — its return value is whatever `Command(resume=...)` sends back
2. **`Command(resume=value)`** continues execution — the node branches on the human's decision
3. A **checkpointer is required** — without saved state, there's nothing to resume from